# XGBoost training: SA-AKI static representation

The first of five model training notebooks.

Input is `data/static_representation.parquet` from preprocessing: one row per
stay, aggregate features, label attached.

Two imbalance-handling conditions are run as complete, independent pipelines,
each with its own Optuna search, refit and test evaluation. The train/test
split is shared between them.

- No imbalance handling: `scale_pos_weight` left at the XGBoost default.
- Class weighting: `scale_pos_weight` computed from training class counts
  at each fit.

SMOTE is out of scope for this notebook.

## Setup

Imports and file paths. The fit/transform functions are imported from
`src/preprocessing_fit_transform.py`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    recall_score, precision_score, f1_score,
)
from xgboost import XGBClassifier

sys.path.insert(0, str(Path.cwd().parent))
from src.preprocessing_fit_transform import (
    winsorize_fit, winsorize_transform,
    impute_fit, impute_transform,
    scale_fit, scale_transform,
    encode,
)

DATA_DIR = Path("../data")
DATA_PATH = DATA_DIR / "static_representation.parquet"

RANDOM_STATE = 42
N_TRIALS = 50
N_FOLDS = 5
N_ESTIMATORS_CEILING = 1000
EARLY_STOPPING_ROUNDS = 30

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("xgboost", __import__("xgboost").__version__, "| optuna", optuna.__version__)

## 1. Load the data

Prints shape and class balance, and checks the column names against what
preprocessing produces.

In [ ]:
df = pd.read_parquet(DATA_PATH)

print("shape:", df.shape)
print("stays:", df["stay_id"].nunique(), "| patients:", df["subject_id"].nunique())
print("\nclass balance:")
print(df["label"].value_counts().rename("count"))
print(f"positive rate: {df['label'].mean() * 100:.2f}%")
df.head()

### Confirm the columns match what is expected

Aggregate feature columns follow the `{feature}_{stat}` naming from
preprocessing, with stats min, max, mean and last. Identifier, categorical
and label columns are checked to be present.

In [ ]:
STATS = ["min", "max", "mean", "last"]
AGG_COLS = [c for c in df.columns if c.rsplit("_", 1)[-1] in STATS]
REQUIRED = ["stay_id", "subject_id", "admission_age", "gender", "admission_type",
            "aki_hour", "label"]

missing = [c for c in REQUIRED if c not in df.columns]
unaccounted = sorted(set(df.columns) - set(AGG_COLS) - set(REQUIRED))

print(f"aggregate feature columns: {len(AGG_COLS)}  (expected 22 features x 4 stats = 88)")
print("missing required columns:", missing if missing else "none")
print("unaccounted columns:", unaccounted if unaccounted else "none")

assert not missing, f"missing required columns: {missing}"
assert not unaccounted, f"unexpected columns: {unaccounted}"

## 2. Separate features from label and identifiers

`label` is the target. `stay_id` and `subject_id` are identifiers and are
excluded from the model. `gender` and `admission_type` are handled by
`encode()`. The remaining numeric columns, the aggregates plus
`admission_age`, are passed to Winsorize, imputation and scaling.

In [ ]:
TARGET = "label"
ID_COLS = ["stay_id", "subject_id"]
CATEGORICAL = ["gender", "admission_type"]
FEATURE_COLS = AGG_COLS + ["admission_age"]   # winsorize/impute/scale apply here

# columns dropped when building the model matrix
# aki_hour is the AKI onset time, which determines the label. It is carried
# for the lead-time analysis and must never reach the model matrix.
LABEL_SUPPORT = ["aki_hour"]
DROP_FROM_X = ID_COLS + LABEL_SUPPORT + [TARGET]

print(f"numeric feature columns (winsorize/impute/scale): {len(FEATURE_COLS)}")
print(f"categorical (encoded): {CATEGORICAL}")
print(f"identifiers (excluded from X): {ID_COLS}")

## 3. Train/test split

80/20, stratified by `label`, `random_state=42`. The static representation is
one row per patient, so no patient-level grouping is applied.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df[TARGET], random_state=RANDOM_STATE
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train: {len(train_df):,} rows, positive rate {train_df[TARGET].mean() * 100:.2f}%")
print(f"test:  {len(test_df):,} rows, positive rate {test_df[TARGET].mean() * 100:.2f}%")
print(f"\npatient overlap between train and test: "
      f"{len(set(train_df['subject_id']) & set(test_df['subject_id']))}")

## 4. Fold processing

Winsorize, imputation and scaling are fitted on the training portion passed to
this function and applied unchanged to the other portion. `encode()` is then
applied to both. The same function serves the inner CV folds and the final
refit.

In [ ]:
def process(fit_df, apply_df, feature_cols=FEATURE_COLS):
    """Fit Winsorize/impute/scale on fit_df, apply to both frames, then encode.

    Returns (fit_df_processed, apply_df_processed, fitted_params).
    """
    bounds = winsorize_fit(fit_df, feature_cols)
    fit_df = winsorize_transform(fit_df, bounds)
    apply_df = winsorize_transform(apply_df, bounds)

    medians = impute_fit(fit_df, feature_cols)
    fit_df = impute_transform(fit_df, medians)
    apply_df = impute_transform(apply_df, medians)

    scaler = scale_fit(fit_df, feature_cols)
    fit_df = scale_transform(fit_df, scaler, feature_cols)
    apply_df = scale_transform(apply_df, scaler, feature_cols)

    return encode(fit_df), encode(apply_df), {
        "bounds": bounds, "medians": medians, "scaler": scaler,
    }


def to_xy(processed_df):
    """Split a processed frame into the model matrix and the target."""
    X = processed_df.drop(columns=DROP_FROM_X)
    y = processed_df[TARGET]
    return X, y


def pos_weight(y):
    """Return negative count over positive count, used for class weighting."""
    return float((y == 0).sum() / (y == 1).sum())

## 5. Optuna objective and search space

Hyperparameters are selected on inner folds and evaluated on a test set held
out of selection (Cawley & Talbot, 2010; Varma & Simon, 2006).

Each trial builds five stratified folds from the training split. Winsorize,
imputation and scaling are fitted on each fold's own training portion. For
class weighting, `scale_pos_weight` is recomputed from that fold's training
labels. Trees are grown to a ceiling of 1000 with early stopping on the
validation fold, monitoring AUPRC.

Early stopping and the evaluation metric are set on the constructor, per the
XGBoost 3.x scikit-learn API, not passed to `fit()`.

Mean fold AUPRC is reported after each fold, allowing a trial to be pruned
before all five folds run. The boosting rounds chosen by early stopping are
stored on the trial for use at refit.

| Parameter | Status | Range / value |
|---|---|---|
| `max_depth` | Tuned | 3 to 10, integer |
| `learning_rate` | Tuned | 0.01 to 0.3, log scale |
| `subsample` | Tuned | 0.5 to 1.0 |
| `colsample_bytree` | Tuned | 0.5 to 1.0 |
| `reg_alpha` | Tuned | 1e-3 to 10, log scale |
| `reg_lambda` | Tuned | 1e-3 to 10, log scale |
| `n_estimators` | Fixed | Ceiling 1000, early stopping |
| `scale_pos_weight` | Fixed | No imbalance handling: unset. Class weighting: computed per fit |
| `min_child_weight`, `gamma` | Excluded | Library defaults |
| `colsample_bylevel`, `colsample_bynode` | Excluded | Library defaults |

In [ ]:
def suggest_params(trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }


def make_objective(condition):
    """Return the Optuna objective for the given imbalance-handling condition."""

    def objective(trial):
        params = suggest_params(trial)
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

        fold_scores, fold_best_iters = [], []
        for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df[TARGET])):
            fold_train = train_df.iloc[tr_idx].reset_index(drop=True)
            fold_valid = train_df.iloc[va_idx].reset_index(drop=True)

            fold_train, fold_valid, _ = process(fold_train, fold_valid)
            X_tr, y_tr = to_xy(fold_train)
            X_va, y_va = to_xy(fold_valid)

            extra = {"scale_pos_weight": pos_weight(y_tr)} if condition == "class weighting" else {}

            model = XGBClassifier(
                n_estimators=N_ESTIMATORS_CEILING,
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,   # constructor, XGBoost 3.x
                eval_metric="aucpr",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                **params,
                **extra,
            )
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

            proba = model.predict_proba(X_va)[:, 1]
            fold_scores.append(average_precision_score(y_va, proba))
            fold_best_iters.append(model.best_iteration)

            trial.report(float(np.mean(fold_scores)), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        trial.set_user_attr("mean_best_iteration", int(np.mean(fold_best_iters)))
        return float(np.mean(fold_scores))

    return objective

## 6. Run the Optuna study, once per condition

50 trials per condition, maximising mean fold AUPRC, with median pruning
(Akiba et al., 2019). The two studies are independent.

In [ ]:
def run_study(condition):
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
        study_name=f"xgboost_{condition.lower().replace(' ', '_')}",
    )
    study.optimize(make_objective(condition), n_trials=N_TRIALS, show_progress_bar=True)

    pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f"\n{condition}: {len(study.trials)} trials, {pruned} pruned")
    print(f"best mean CV AUPRC: {study.best_value:.4f}")
    print(f"best boosting rounds (mean over folds): "
          f"{study.best_trial.user_attrs['mean_best_iteration']}")
    print("best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    return study

### No imbalance handling

In [ ]:
study_a = run_study("no imbalance handling")

### Class weighting

In [ ]:
study_b = run_study("class weighting")

## 7. Refit on the full training split

For each condition, Winsorize, imputation and scaling are fitted once on the
entire 80% training split. One model per condition is then trained on that
processed data using the condition's winning hyperparameters.

No validation fold is used at this stage. The number of boosting rounds is
taken from the mean best iteration recorded during that condition's
cross-validation.

In [ ]:
def refit(study, condition):
    params = dict(study.best_params)
    n_rounds = max(1, study.best_trial.user_attrs["mean_best_iteration"] + 1)

    train_proc, test_proc, fitted = process(train_df.copy(), test_df.copy())
    X_tr, y_tr = to_xy(train_proc)

    extra = {"scale_pos_weight": pos_weight(y_tr)} if condition == "class weighting" else {}

    model = XGBClassifier(
        n_estimators=n_rounds,          # no validation fold, so no early stopping
        eval_metric="aucpr",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
        **extra,
    )
    model.fit(X_tr, y_tr, verbose=False)

    print(f"{condition}: refit on {len(X_tr):,} rows, {n_rounds} boosting rounds"
          + (f", scale_pos_weight {extra['scale_pos_weight']:.4f}" if extra else ""))
    return model, test_proc, fitted


model_a, test_proc_a, fitted_a = refit(study_a, "no imbalance handling")
model_b, test_proc_b, fitted_b = refit(study_b, "class weighting")

## 8. Evaluate on the held-out test set

The Winsorize bounds, medians and scaler fitted in step 7 are applied to the
test set unchanged. Nothing is refitted on test and no resampling is applied
to it. AUPRC is the primary metric, with AUROC, recall, precision and F1
reported at a 0.5 decision threshold.

In [ ]:
def evaluate(model, test_proc, condition):
    X_te, y_te = to_xy(test_proc)
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)

    return {
        "condition": condition,
        "AUPRC": average_precision_score(y_te, proba),
        "AUROC": roc_auc_score(y_te, proba),
        "recall_AKI": recall_score(y_te, pred),                    # positive class
        "specificity_non_AKI": recall_score(y_te, pred, pos_label=0),  # negative class
        "precision": precision_score(y_te, pred),
        "F1": f1_score(y_te, pred),
    }


metrics_a = evaluate(model_a, test_proc_a, "no imbalance handling")
metrics_b = evaluate(model_b, test_proc_b, "class weighting")

print(f"test rows: {len(test_proc_a):,}")
print(f"test positive rate: {test_df[TARGET].mean() * 100:.2f}%  "
      f"(AUPRC of a random baseline equals the positive rate)")

## 9. Calibration of the predicted probabilities

Calibration compares predicted probabilities against the rates observed at
those probabilities, where AUPRC and AUROC measure ranking.

Predictions come from the models refitted in step 7. Nothing is refitted,
retrained or resampled here. Both conditions are drawn on one chart.

In [ ]:
from src.evaluation_utils import calibration_report, plot_calibration

# processed test frames and models from steps 7 and 8
X_test_a, y_test = to_xy(test_proc_a)
X_test_b, _ = to_xy(test_proc_b)

proba_a = model_a.predict_proba(X_test_a)[:, 1]
proba_b = model_b.predict_proba(X_test_b)[:, 1]

bins_a = calibration_report(y_test, proba_a, label="no imbalance handling")
bins_b = calibration_report(y_test, proba_b, label="class weighting")

fig = plot_calibration([(bins_a, "no imbalance handling"),
                        (bins_b, "class weighting")])

## 10. Bootstrap confidence intervals

The test set is resampled with replacement and each metric from step 8 is
recomputed on every resample, giving an interval around each point estimate.

Predictions come from the models refitted in step 7. Only the test set rows
are resampled.

In [ ]:
from IPython.display import display

from src.evaluation_utils import bootstrap_metrics_report

# proba_a, proba_b and y_test come from step 9
boot_a = bootstrap_metrics_report(y_test, proba_a, threshold=0.5,
                                  label="no imbalance handling")
boot_b = bootstrap_metrics_report(y_test, proba_b, threshold=0.5,
                                  label="class weighting")

display(boot_a)
display(boot_b)

## 11. Subgroup performance

Performance is reported separately by gender, age band and admission type,
each with the bootstrap intervals used in step 10. Group sizes are reported
alongside, and subgroup intervals are wider than the overall result.

Predictions come from the models refitted in step 7. Nothing is refitted or
retrained.

In [ ]:
from IPython.display import display

from src.evaluation_utils import subgroup_report

# Subgroup labels come from the untransformed test split: admission_age in
# test_proc_* is scaled, so age banding needs the raw values. Row order is
# unchanged, so these line up with y_test, proba_a and proba_b.
gender_values = test_df["gender"]

age_bands = pd.cut(test_df["admission_age"], bins=[18, 40, 65, 90],
                   labels=["18-40", "40-65", "65-90"], include_lowest=True)

# the encoded emergency/elective split; several raw admission_type
# categories are too small to bootstrap
admission_groups = test_proc_a["admission_type_elective"].map(
    {0: "Emergency", 1: "Elective"}
)

print("group sizes")
for name, values in [("gender", gender_values), ("age band", age_bands),
                     ("admission type", admission_groups)]:
    counts = values.value_counts(dropna=False).sort_index().to_dict()
    print(f"  {name}: {counts}")
print()

In [ ]:
SUBGROUPS = {
    "Gender": gender_values,
    "Age band": age_bands,
    "Admission type": admission_groups,
}
CONDITIONS = {"no imbalance handling": proba_a, "class weighting": proba_b}

subgroup_tables = {}
for attribute, values in SUBGROUPS.items():
    for condition, proba in CONDITIONS.items():
        key = f"{condition} -- {attribute}"
        subgroup_tables[key] = subgroup_report(y_test, proba, values, label=key)

### Subgroup results

One table per condition and attribute, each row a metric within one subgroup,
with the point estimate, the confidence interval and the subgroup size.

In [ ]:
for key, table in subgroup_tables.items():
    print(key)
    display(table)

## 12. Decision curve analysis

Net benefit is computed across the full range of thresholds and compared
against two reference strategies, flagging every patient and flagging none
(Vickers & Elkin, 2006). A model is ahead over the range where its curve sits
above both reference lines.

Predictions come from the models refitted in step 7. Nothing is refitted or
retrained.

In [ ]:
from src.evaluation_utils import decision_curve, plot_decision_curve

# y_test, proba_a and proba_b come from step 9
curve_a = decision_curve(y_test, proba_a)
curve_b = decision_curve(y_test, proba_b)

fig_dca = plot_decision_curve([(curve_a, "no imbalance handling"),
                              (curve_b, "class weighting")])

print(curve_a[(curve_a["threshold"] >= 0.4) & (curve_a["threshold"] <= 0.7)].round(4).to_string(index=False))
print()
print(curve_b[(curve_b["threshold"] >= 0.4) & (curve_b["threshold"] <= 0.7)].round(4).to_string(index=False))

## 13. Summary table

One row per condition.

In [ ]:
summary = pd.DataFrame([metrics_a, metrics_b]).set_index("condition").round(4)
summary

## Feature importance

XGBoost exposes a native gain-based importance, so no permutation is needed.
Gain is the average improvement in the split criterion each feature brings
across the trees it appears in. A feature never used in a split scores zero.

Importance is reported per engineered column, not aggregated per clinical
variable. Aggregation belongs in the cross-model comparison, applied
consistently across models.

No model is retrained or refitted here. Both conditions use the models and test
sets already built above.

In [ ]:
def native_importance(model, feature_names):
    """Return the gain-based importance of every feature, zero-filled."""
    gains = model.get_booster().get_score(importance_type="gain")
    frame = pd.DataFrame({
        "feature": list(feature_names),
        "importance": [float(gains.get(f, 0.0)) for f in feature_names],
    })
    return frame.sort_values("importance", ascending=False).reset_index(drop=True)


FEATURE_NAMES = list(to_xy(test_proc_a)[0].columns)

importance_tables = {
    "no imbalance handling": native_importance(model_a, FEATURE_NAMES),
    "class weighting": native_importance(model_b, FEATURE_NAMES),
}

for condition, table in importance_tables.items():
    used = int((table["importance"] > 0).sum())
    print(f"{condition}: {used} of {len(table)} features used in at least one split")
    print(table.head(10).round(4).to_string(index=False))
    print()

### Figure: feature importance

The ten highest-ranked features, with both conditions shown together. Features
are ranked by the no-imbalance-handling condition, and the same features are
shown for both, so the bars line up for comparison.

In [ ]:
from src.evaluation_utils import plot_feature_importance

fig_importance = plot_feature_importance(
    [(table["feature"], table["importance"], condition)
     for condition, table in importance_tables.items()],
    top_n=10,
    title="Feature importance: XGBoost",
)

### Save the importance tables

Written to `data/`, one file per model, holding both conditions and the feature
order so the cross-model comparison can align them.

In [ ]:
importance_path = DATA_DIR / "xgboost_feature_importance.json"
importance_path.write_text(json.dumps({
    "model": "xgboost",
    "method": "native",
    "metric": "gain",
    "feature_names": FEATURE_NAMES,
    "importance": {c: t.set_index("feature")["importance"].to_dict()
                   for c, t in importance_tables.items()},
}, indent=2))
print(f"wrote {importance_path.name}  ({importance_path.stat().st_size / 1024:.1f} KB)")

## Output

Per condition: winning hyperparameters, test metrics, and the fitted
Winsorize bounds, medians and scaler parameters. Written to `data/`, which is
gitignored. The scaler is stored as its centre and scale arrays rather than
the object itself.

In [ ]:
def save_results(study, metrics, fitted, calibration_bins, bootstrap_table,
                 decision_curve_table, subgroup_tables, condition):
    payload = {
        "condition": condition,
        "best_params": study.best_params,
        "n_estimators_used_at_refit": study.best_trial.user_attrs["mean_best_iteration"] + 1,
        "best_cv_auprc": study.best_value,
        "test_metrics": {k: v for k, v in metrics.items() if k != "condition"},
        # subgroup point estimates, keyed by attribute then subgroup, so the
        # comparison notebook can chart them without recomputing
        "subgroups": {
            key.split(" -- ", 1)[1]: {
                str(subgroup): dict(zip(group["metric"], group["point_estimate"]))
                for subgroup, group in table.groupby("subgroup")
            }
            for key, table in subgroup_tables.items()
            if key.startswith(condition + " -- ")
        },
        # net benefit per threshold, from the decision curve computed above.
        # "treat no one" is 0 at every threshold by definition, so it is not stored.
        "decision_curve": {
            "threshold": [float(v) for v in decision_curve_table["threshold"]],
            "net_benefit_model": [float(v) for v in decision_curve_table["net_benefit_model"]],
            "net_benefit_all": [float(v) for v in decision_curve_table["net_benefit_all"]],
        },
        "calibration": {
            "brier": float(calibration_bins.attrs["brier"]),
            "ece": float(calibration_bins.attrs["ece"]),
            "mce": float(calibration_bins.attrs["mce"]),
        },
        "confidence_intervals": {
            row["metric"]: {"ci_lower": float(row["ci_lower"]),
                            "ci_upper": float(row["ci_upper"])}
            for _, row in bootstrap_table.iterrows()
        },
        "fitted_params": {
            "winsorize_bounds": {k: [float(lo), float(hi)] for k, (lo, hi) in fitted["bounds"].items()},
            "medians": {k: float(v) for k, v in fitted["medians"].items()},
            "scaler_center": [float(v) for v in fitted["scaler"].center_],
            "scaler_scale": [float(v) for v in fitted["scaler"].scale_],
            "scaler_feature_order": FEATURE_COLS,
        },
    }
    path = DATA_DIR / f"xgboost_{condition.lower().replace(' ', '_')}_results.json"
    path.write_text(json.dumps(payload, indent=2))
    print(f"wrote {path.name}  ({path.stat().st_size / 1024:.1f} KB)")


save_results(study_a, metrics_a, fitted_a, bins_a, boot_a, curve_a,
             subgroup_tables, "no imbalance handling")
save_results(study_b, metrics_b, fitted_b, bins_b, boot_b, curve_b,
             subgroup_tables, "class weighting")